In [150]:
import pandas as pd
import re
import requests
from datetime import date, timedelta
import unicodedata
import json
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("display.max_colwidth", None)

In [151]:
from unicodedata import normalize as uninorm

def normalize_string(s):
    if not isinstance(s, str):
        return s
    s = s.lower()
    s = uninorm("NFD", s)
    s = "".join(c for c in s if __import__('unicodedata').category(c) != "Mn")
    s = " ".join(s.split())
    return s
# True

In [152]:
df = pd.read_csv("../data/raw_procurements.csv")
DEFAULT_CPVS: dict[str, str] = {
    "75251100-1": "Υπηρεσίες καταπολέμησης πυρκαγιών",
    "75251110-4": "Υπηρεσίες πρόληψης πυρκαγιών",
    "75251120-7": "Υπηρεσίες καταπολέμησης δασοπυρκαγιών",
    "77200000-2": "Υπηρεσίες δασοκομίας",
    "77312000-0": "Υπηρεσίες εκκαθάρισης από αγριόχορτα",
    "77314000-4": "Υπηρεσίες συντήρησης οικοπέδων",
    "77340000-5": "Κλάδεμα δένδρων και θάμνων",
    "45343000-3": "Εργασίες εγκαταστάσεων πρόληψης πυρκαγιάς",
    "77314100-5": "Υπηρεσίες χορτοκάλυψης",
    "34144200-0": "Οχήματα για τις υπηρεσίες εκτάκτου ανάγκης",
    "34144212-7": "Υδροφόρες πυροσβεστικών οχημάτων",
    "35111000-5": "Πυροσβεστικός εξοπλισμός",
    "35111200-7": "Υλικά πυρόσβεσης",
    "44480000-8": "Ποικίλος εξοπλισμός πυροπροστασίας"
}

In [149]:
df[df.organization_value.str.contains("Λαγκαδ", case=False)]

,title,referenceNumber,submissionDate,contractSignedDate,startDate,noEndDate,endDate,cancelled,cancellationDate,cancellationType,cancellationReason,decisionRelatedAda,contractNumber,organizationVatNumber,greekOrganizationVatNumber,diavgeiaADA,budget,contractBudget,bidsSubmitted,maxBidsSubmitted,numberOfSections,centralGovernmentAuthority,nutsCode_key,nutsCode_value,organization_key,organization_value,procedureType_key,procedureType_value,awardProcedure,nutsCity,nutsPostalCode,centralizedMarkets,contractType,assignCriteria,classificationOfPublicLawOrganization,typeOfContractingAuthority,contractingAuthorityActivity,contractDuration,contractDurationUnitOfMeasure,contractRelatedADA,fundingDetails_cofund,fundingDetails_selfFund,fundingDetails_espa,fundingDetails_regularBudget,unitsOperator,signers,firstMember_vatNumber,firstMember_name,totalCostWithVAT,totalCostWithoutVAT,shortDescriptions,cpv_keys,cpv_values,greenContracts,auctionRefNo,paymentRefNo
4006,ΠΡΟΜΗΘΕΙΑΣ ΕΝΟΣ ΠΥΡΟΣΒΕΣΤΙΚΟΥ ΕΞΟΠΛΙΣΜΟΥ -ΤΜΗΜΑ 3,26SYMV018473491,2026-02-12T13:20:49.151,2026-02-12,2026-02-12,False,2026-10-12,False,NaN,NaN,NaN,NaN,3843,998246659,True,NaN,NaN,290322.58,NaN,1.0,3.0,Όχι,EL522,Θεσσαλονίκη,6165,ΔΗΜΟΣ ΛΑΓΚΑΔΑ,1.0,Ανοιχτή διαδικασία,NaN,ΛΑΓΚΑΔΑ,572 00,Δεν χρησιμοποιείται,Προμήθειες,Βάσει τιμής,ΟΤΑ,ΝΠΔΔ,Γενικές δημόσιες υπηρεσίες,8.0,Μήνες,9ΞΘΔΩΛΛ-3ΟΝ,NaN,NaN,NaN,NaN,ΠΡΟΜΗΘΕΙΩΝ,ΕΛΠΙΝΙΚΗ ΑΝΔΡΕΑΔΟΥ - Δήμαρχος,801083010,C. I. PARTS Ι.Κ.Ε.,17360.0,14000.0,ΠΡΟΜΗΘΕΙΑΣ ΕΝΟΣ ΠΥΡΟΣΒΕΣΤΙΚΟΥ ΕΞΟΠΛΙΣΜΟΥ -ΤΜΗΜΑ 3,35111000-5,Πυροσβεστικός εξοπλισμός,Δεν εμπίπτει στο ΕΣΔ,25AWRD018061107,[]
4009,"ΠΡΟΜΗΘΕΙΑΣ ΕΝΟΣ (1) ΒΥΤΙΟΦΟΡΟΥ ΟΧΗΜΑΤΟΣ, ΣΤΟ ΔΗΜΟ ΛΑΓΚΑΔΑ- ΤΜΗΜΑ 1",26SYMV018491493,2026-02-17T09:45:20.6,2026-02-16,2026-02-16,False,2026-08-16,False,NaN,NaN,NaN,NaN,4163,998246659,True,NaN,NaN,290322.58,NaN,1.0,3.0,Όχι,EL522,Θεσσαλονίκη,6165,ΔΗΜΟΣ ΛΑΓΚΑΔΑ,1.0,Ανοιχτή διαδικασία,NaN,ΛΑΓΚΑΔΑ,572 00,Δεν χρησιμοποιείται,Προμήθειες,Βάσει τιμής,ΟΤΑ,ΝΠΔΔ,Γενικές δημόσιες υπηρεσίες,6.0,Μήνες,9ΞΘΔΩΛΛ-3ΟΝ,NaN,NaN,NaN,NaN,ΠΡΟΜΗΘΕΙΩΝ,ΕΛΠΙΝΙΚΗ ΑΝΔΡΕΑΔΟΥ - Δήμαρχος,094270233,ΕΛΛΗΝΙΚΗ ΒΙΟΜΗΧΑΝΙΑ ΠΕΡΙΒΑΛΟΝΤΙΚΩΝ ΣΥΣΤΗΜΑΤΩΝ-ΑΝΩΝΥΜΗ ΕΜΠΟΡΙΚΗ - ΒΙΟΜΗΧΑΝΙΚΗ ΕΤΑΙΡΕΙΑ ΑΕΒΕ,227912.0,183800.0,"ΠΡΟΜΗΘΕΙΑΣ ΕΝΟΣ (1) ΒΥΤΙΟΦΟΡΟΥ ΟΧΗΜΑΤΟΣ, ΣΤΟ ΔΗΜΟ ΛΑΓΚΑΔΑ, ΠΟΣΟΥ 227.912,00 με Φ.Π.Α. 24%- ΤΜΗΜΑ 1",34144212-7,Υδροφόρες πυροσβεστικών οχημάτων,Δεν εμπίπτει στο ΕΣΔ,25AWRD018061107,[]


In [175]:
df.contractType.value_counts()

contractType
Υπηρεσίες                               3119
Προμήθειες                               713
Έργα                                     125
Τεχνικές ή λοιπές συναφείς υπηρεσίες      54
Μελέτες                                   10
Name: count, dtype: int64

In [179]:
pd.options.display.float_format = '{:,.2f}'.format
df.groupby("organization_value")['totalCostWithoutVAT'].sum().sort_values(ascending=False)

organization_value
ΥΠΟΥΡΓΕΙΟ ΠΕΡΙΒΑΛΛΟΝΤΟΣ ΚΑΙ ΕΝΕΡΓΕΙΑΣ                                                                                      438,269,333.89
ΥΠΟΥΡΓΕΙΟ ΚΛΙΜΑΤΙΚΗΣ ΚΡΙΣΗΣ ΚΑΙ ΠΟΛΙΤΙΚΗΣ ΠΡΟΣΤΑΣΙΑΣ                                                                       159,835,424.63
ΚΟΙΝΩΝΙΑ ΤΗΣ ΠΛΗΡΟΦΟΡΙΑΣ ΑΕ                                                                                                 25,768,800.00
ΠΕΡΙΦΕΡΕΙΑ ΑΤΤΙΚΗΣ                                                                                                          17,849,223.09
ΔΗΜΟΣ ΗΛΙΟΥΠΟΛΗΣ                                                                                                            14,174,847.13
ΑΝΕΞΑΡΤΗΤΟΣ ΔΙΑΧΕΙΡΙΣΤΗΣ ΜΕΤΑΦΟΡΑΣ ΗΛΕΚΤΡΙΚΗΣ ΕΝΕΡΓΕΙΑΣ ΑΔΜΗΕ ΑΕ                                                             9,045,258.58
ΔΗΜΟΣ ΒΑΡΗΣ - ΒΟΥΛΑΣ - ΒΟΥΛΙΑΓΜΕΝΗΣ                                                                                          5,833,607.78
ΕΥΔΑΠ ΑΕ       

In [161]:
df=df.drop_duplicates(keep="first")

In [162]:
# df[df.referenceNumber.notnull()]

In [163]:
df.duplicated().value_counts()

False    4021
Name: count, dtype: int64

In [154]:
df['contractSignedDate'] = pd.to_datetime(df['contractSignedDate'])
df = df.sort_values(by = 'contractSignedDate', ascending=True)
latest_date = df.tail(1)['contractSignedDate'].item()
one_year_before = latest_date - pd.DateOffset(years=1)
two_year_before = latest_date - pd.DateOffset(years=2)
print(f"latest date in the data: {latest_date}. \nOne year before = {one_year_before}. \nTwo years before = {two_year_before}")
df.tail(1)

latest date in the data: 2026-02-27 00:00:00. 
One year before = 2025-02-27 00:00:00. 
Two years before = 2024-02-27 00:00:00


,title,referenceNumber,submissionDate,contractSignedDate,startDate,noEndDate,endDate,cancelled,cancellationDate,cancellationType,cancellationReason,decisionRelatedAda,contractNumber,organizationVatNumber,greekOrganizationVatNumber,diavgeiaADA,budget,contractBudget,bidsSubmitted,maxBidsSubmitted,numberOfSections,centralGovernmentAuthority,nutsCode_key,nutsCode_value,organization_key,organization_value,procedureType_key,procedureType_value,awardProcedure,nutsCity,nutsPostalCode,centralizedMarkets,contractType,assignCriteria,classificationOfPublicLawOrganization,typeOfContractingAuthority,contractingAuthorityActivity,contractDuration,contractDurationUnitOfMeasure,contractRelatedADA,fundingDetails_cofund,fundingDetails_selfFund,fundingDetails_espa,fundingDetails_regularBudget,unitsOperator,signers,firstMember_vatNumber,firstMember_name,totalCostWithVAT,totalCostWithoutVAT,shortDescriptions,cpv_keys,cpv_values,greenContracts,auctionRefNo,paymentRefNo
8041,"ΣΥΜΒΑΣΗ ΕΥΡΩ : # 37.200,00€ #.-",26SYMV018554537,2026-02-27T19:19:46.033,2026-02-27,2026-02-28,False,2027-01-24,False,NaN,NaN,NaN,NaN,Α 02/2026,997881842,False,NaN,NaN,30000.0,NaN,1.0,8.0,Ναι,EL306,Δυτική Αττική,100015969,ΥΠΟΥΡΓΕΙΟ ΝΑΥΤΙΛΙΑΣ ΚΑΙ ΝΗΣΙΩΤΙΚΗΣ ΠΟΛΙΤΙΚΗΣ,6.0,Απευθείας ανάθεση,NaN,ΑΣΠΡΟΠΥΡΓΟΣ,193 00,Δυναμικό σύστημα αγορών,Υπηρεσίες,Βάσει τιμής,Κεντρική Κυβέρνηση,Κεντρική Διοίκηση,Εκπαίδευση,12.0,Μήνες,62ΩΒ4653ΠΩ-6ΞΧ,NaN,0878/(Α),NaN,NaN,ΑΕΝ ΑΣΠΡΟΠΥΡΓΟΥ,ΙΩΑΝΝΗΣ ΜΥΤΙΛΗΝΑΙΟΣ - Λιμενάρχης,802150294,ΣΚΑΦΙΔΑΣ ΕΥΣΤΡΑΤΙΟΣ Ε.Ε.,37200.0,30000.0,Παροχή υπηρεσιών αποψίλωσης - κλαδονομής - καλλωπισμού και αποκομιδής καταλοίπων περιβάλλοντος χώρου Α.Ε.Ν./Ασπροπύργου διάρκειας δώδεκα (12) μηνών.- | Παροχή υπηρεσιών αποψίλωσης - κλαδονομής - καλλωπισμού και αποκομιδής καταλοίπων περιβάλλοντος χώρου Α.Ε.Ν./Ασπροπύργου διάρκειας δώδεκα (12) μηνών.- | Παροχή υπηρεσιών αποψίλωσης - κλαδονομής - καλλωπισμού και αποκομιδής καταλοίπων περιβάλλοντος χώρου Α.Ε.Ν./Ασπροπύργου διάρκειας δώδεκα (12) μηνών.- | Παροχή υπηρεσιών αποψίλωσης - κλαδονομής - καλλωπισμού και αποκομιδής καταλοίπων περιβάλλοντος χώρου Α.Ε.Ν./Ασπροπύργου διάρκειας δώδεκα (12) μηνών.- | Παροχή υπηρεσιών αποψίλωσης - κλαδονομής - καλλωπισμού και αποκομιδής καταλοίπων περιβάλλοντος χώρου Α.Ε.Ν./Ασπροπύργου διάρκειας δώδεκα (12) μηνών.- | Παροχή υπηρεσιών αποψίλωσης - κλαδονομής - καλλωπισμού και αποκομιδής καταλοίπων περιβάλλοντος χώρου Α.Ε.Ν./Ασπροπύργου διάρκειας δώδεκα (12) μηνών.- | Παροχή υπηρεσιών αποψίλωσης - κλαδονομής - καλλωπισμού και αποκομιδής καταλοίπων περιβάλλοντος χώρου Α.Ε.Ν./Ασπροπύργου διάρκειας δώδεκα (12) μηνών.- | Παροχή υπηρεσιών αποψίλωσης - κλαδονομής - καλλωπισμού και αποκομιδής καταλοίπων περιβάλλοντος χώρου Α.Ε.Ν./Ασπροπύργου διάρκειας δώδεκα (12) μηνών.-,77310000-6 | 77312000-0 | 77342000-9 | 77341000-2 | 77341000-2 | 77314000-4 | 77231200-0 | 77314000-4,Φύτευση και συντήρηση χώρων πρασίνου | Υπηρεσίες εκκαθάρισης από αγριόχορτα | Κλάδεμα θάμνων | Κλάδεμα δένδρων | Κλάδεμα δένδρων | Υπηρεσίες συντήρησης οικοπέδων | Υπηρεσίες ελέγχου παρασίτων του δάσους | Υπηρεσίες συντήρησης οικοπέδων,Δεν εμπίπτει στο ΕΣΔ | Δεν εμπίπτει στο ΕΣΔ | Δεν εμπίπτει στο ΕΣΔ | Δεν εμπίπτει στο ΕΣΔ | Δεν εμπίπτει στο ΕΣΔ | Δεν εμπίπτει στο ΕΣΔ | Δεν εμπίπτει στο ΕΣΔ | Δεν εμπίπτει στο ΕΣΔ,26AWRD018554440,[]


In [124]:
# missing dates - false = δεν λείπουν ημερομηνίες
df.contractSignedDate.isnull().value_counts()

contractSignedDate
False    4021
Name: count, dtype: int64

In [125]:
# hero section data validation

In [126]:
df_2026 = df[df['contractSignedDate'].between(pd.to_datetime("2026-01-01"), latest_date)]
df_2025 = df[df['contractSignedDate'].between(pd.to_datetime("2025-01-01"), one_year_before)]
df_2024 = df[df['contractSignedDate'].between(pd.to_datetime("2024-01-01"), two_year_before)]
print(f"{df_2026.shape[0]} εγγραφές από 1/1/2026 έως {latest_date}\n{df_2025.shape[0]} εγγραφές από 1/1/2025 έως {one_year_before}\n{df_2024.shape[0]} εγγραφές από 1/1/2024 έως {two_year_before}")

47 εγγραφές από 1/1/2026 έως 2026-02-27 00:00:00
91 εγγραφές από 1/1/2025 έως 2025-02-27 00:00:00
116 εγγραφές από 1/1/2024 έως 2024-02-27 00:00:00


In [132]:
spent_2026 = df_2026.totalCostWithoutVAT.sum().item()
spent_2025 = df_2025.totalCostWithoutVAT.sum().item()
spent_2024 = df_2024.totalCostWithoutVAT.sum().item()
change_cmp_2025 = round(((spent_2026 - spent_2025)/spent_2025)*100,1)
change_cmp_2024 = round(((spent_2026 - spent_2024)/spent_2024)*100,1)
print(f"Up until {latest_date} were spent {df_2026.totalCostWithoutVAT.sum().item()} euros\nThe same timeframe in 2025 were spent {df_2025.totalCostWithoutVAT.sum().item()}\nThe same timeframe in 2024 were spent {df_2024.totalCostWithoutVAT.sum().item()}")
print(f"In 2026 have been spent {change_cmp_2025}% compared to 2025 and {change_cmp_2024}% compared to 2024")

Up until 2026-02-27 00:00:00 were spent 9812963.5 euros
The same timeframe in 2025 were spent 54833220.27
The same timeframe in 2024 were spent 74908518.51999998
In 2026 have been spent -82.1% compared to 2025 and -86.9% compared to 2024


In [133]:
# Συχνότερη διαδικασία ανάθεσης
vc_2026 = df_2026.procedureType_value.value_counts()
top_type_2026 = vc_2026.idxmax()
top_count_2026 = vc_2026.max()

vc_2025 = df_2025.procedureType_value.value_counts()
top_type_2025 = vc_2025.idxmax()
top_count_2025 = vc_2025.max()

vc_2024 = df_2024.procedureType_value.value_counts()
top_type_2024 = vc_2024.idxmax()
top_count_2024 = vc_2024.max()

print(f"In 2026 the most common procedure type is {top_type_2026} with {top_count_2026} contracts\nIn 2025 the most common procedure type is {top_type_2025} with {top_count_2025} contracts\nIn 2024 the most common procedure type is {top_type_2024} with {top_count_2024} contracts")

In 2026 the most common procedure type is Απευθείας ανάθεση with 17 contracts
In 2025 the most common procedure type is Απευθείας ανάθεση (αρ.118/αρ. 328) with 50 contracts
In 2024 the most common procedure type is Απευθείας ανάθεση (αρ.118/αρ. 328) with 64 contracts


In [134]:
# ContractsPage
today = pd.Timestamp.today()
table_data = df[df['contractSignedDate'].between(latest_date-pd.DateOffset(days=27), today)].sort_values(by='contractSignedDate', ascending=False)[["contractSignedDate","organization_value", "title", "cpv_values", "firstMember_name", "procedureType_value", "totalCostWithoutVAT"]]
print(f"table data has {table_data.shape[0]} rows. Spanning from {latest_date-pd.DateOffset(days=27)} to {today}")
table_data.head(1)

table data has 28 rows. Spanning from 2026-01-31 00:00:00 to 2026-03-02 11:59:37.079441


,contractSignedDate,organization_value,title,cpv_values,firstMember_name,procedureType_value,totalCostWithoutVAT
4020,2026-02-27,ΥΠΟΥΡΓΕΙΟ ΝΑΥΤΙΛΙΑΣ ΚΑΙ ΝΗΣΙΩΤΙΚΗΣ ΠΟΛΙΤΙΚΗΣ,"ΣΥΜΒΑΣΗ ΕΥΡΩ : # 37.200,00€ #.-",Φύτευση και συντήρηση χώρων πρασίνου | Υπηρεσίες εκκαθάρισης από αγριόχορτα | Κλάδεμα θάμνων | Κλάδεμα δένδρων | Κλάδεμα δένδρων | Υπηρεσίες συντήρησης οικοπέδων | Υπηρεσίες ελέγχου παρασίτων του δάσους | Υπηρεσίες συντήρησης οικοπέδων,ΣΚΑΦΙΔΑΣ ΕΥΣΤΡΑΤΙΟΣ Ε.Ε.,Απευθείας ανάθεση,30000.0


In [142]:
df_2026[df_2026.procedureType_value=="Απευθείας ανάθεση (αρ.118/αρ. 328)"]

,title,referenceNumber,submissionDate,contractSignedDate,startDate,noEndDate,endDate,cancelled,cancellationDate,cancellationType,cancellationReason,decisionRelatedAda,contractNumber,organizationVatNumber,greekOrganizationVatNumber,diavgeiaADA,budget,contractBudget,bidsSubmitted,maxBidsSubmitted,numberOfSections,centralGovernmentAuthority,nutsCode_key,nutsCode_value,organization_key,organization_value,procedureType_key,procedureType_value,awardProcedure,nutsCity,nutsPostalCode,centralizedMarkets,contractType,assignCriteria,classificationOfPublicLawOrganization,typeOfContractingAuthority,contractingAuthorityActivity,contractDuration,contractDurationUnitOfMeasure,contractRelatedADA,fundingDetails_cofund,fundingDetails_selfFund,fundingDetails_espa,fundingDetails_regularBudget,unitsOperator,signers,firstMember_vatNumber,firstMember_name,totalCostWithVAT,totalCostWithoutVAT,shortDescriptions,cpv_keys,cpv_values,greenContracts,auctionRefNo,paymentRefNo


In [109]:
table_data.duplicated().sum()

np.int64(0)

In [110]:
df.duplicated().sum()

np.int64(0)